# AIMNet2 foundation models: verifying `AIMNet2.from_foundation()` against the `aimnet` package

The xnn [`AIMNet2`](../../src/xnn/gnn/models/aimnet2.py) is written from the
paper (Anstine, Zubatyuk and Isayev, *Chem. Sci.* **16**, 10228, 2025) on the
xnn GNN abstractions, with the conventions of the reference code
[isayevlab/aimnetcentral](https://github.com/isayevlab/aimnetcentral) where the
paper leaves them open. The published checkpoints (six families, four ensemble
members each) are entries of the xnn model hub in the `aimnet2` format: the
loader reads the self-describing artifact (state dict, architecture YAML,
metadata), builds the xnn twin and transplants the weights; the two-body D3(BJ)
term the reference calculator adds post hoc becomes the config's
`subtracted_dispersion` and is served through xnn's own `D3Dispersion`.

This notebook checks that conversion against the reference package itself, on
every family:

1. the registry and the converted architecture;
2. energies, forces and partial charges of the bare networks (no dispersion),
   where the only differences are float32 round-off;
3. the served models, which adds xnn's D3 against the reference D3 kernel;
4. a periodic CO2 box with the damped shifted-force Coulomb sum (the paper's
   setting for its condensed-phase simulation);
5. the Ewald and particle-mesh Ewald Coulomb sums, first the kernels alone on
   fixed charges (against the `nvalchemiops` kernels the reference calls, in
   float32 and float64), then the served models end to end in both precisions;
6. the TorchScript artifact of the served model against the eager model.

The reference package (`pip install aimnet`) is needed only here; xnn's AIMNet2
runs without it.


## 0. Setup

In [1]:
import warnings
warnings.filterwarnings("ignore")
import contextlib
import io
import json
import time

import numpy as np
import torch
import matplotlib.pyplot as plt
from ase import Atoms
from ase.build import molecule

import xnn
from xnn.common.data import structure_to_graph
from xnn.common.models import ForceStressOutput, from_pretrained, list_models, model_card
from xnn.gnn.models import AIMNet2

torch.manual_seed(0)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"xnn: {xnn.__version__} | torch: {torch.__version__} | device: {DEVICE}")

from aimnet.calculators import AIMNet2Calculator
import aimnet
print("reference package aimnet", getattr(aimnet, "__version__", "?"))


xnn: 0.5.0 | torch: 2.8.0+cu126 | device: cuda


reference package aimnet 0.2.0


## 1. The registry

Every published member is a registry entry (`aimnet2-<family>-<member>`); the
reference package's short names resolve as aliases to member 0 of each family.


In [2]:
rows = list_models(architecture="aimnet2", details=True)
print(f"{'name':28s} {'elements':>8s} {'cutoff':>6s}  description")
for r in rows:
    print(f"{r['name']:28s} {r['elements']:>8} {r['cutoff']:>6}  {r['description']}")
print()
for alias in ("aimnet2", "aimnet2-b973c", "aimnet2-2025", "aimnet2-nse", "aimnet2-pd", "aimnet2-rxn"):
    print(f"{alias:16s} -> {model_card(alias).name}")


name                         elements cutoff  description
aimnet2-b973c-2025-d3-0            14    5.0  AIMNet2-2025 B97-3c model with improved intermolecular interactions, ensemble member 0
aimnet2-b973c-2025-d3-1            14    5.0  AIMNet2-2025 B97-3c model with improved intermolecular interactions, ensemble member 1
aimnet2-b973c-2025-d3-2            14    5.0  AIMNet2-2025 B97-3c model with improved intermolecular interactions, ensemble member 2
aimnet2-b973c-2025-d3-3            14    5.0  AIMNet2-2025 B97-3c model with improved intermolecular interactions, ensemble member 3
aimnet2-b973c-d3-0                 14    5.0  AIMNet2 B97-3c general organic model (the first-generation training set), ensemble member 0
aimnet2-b973c-d3-1                 14    5.0  AIMNet2 B97-3c general organic model (the first-generation training set), ensemble member 1
aimnet2-b973c-d3-2                 14    5.0  AIMNet2 B97-3c general organic model (the first-generation training set), ensemble membe

## 2. The converted architecture

`from_pretrained("aimnet2")` returns the served potential: the network inside
xnn's `D3Dispersion` (15 A two-body D3(BJ) with a 3 A switching window, the
reference calculator's post-hoc term), wrapped for forces. `dispersion=False`
gives the bare network. The per-element energy shifts stay float64 while the
network is float32, as in the reference.


In [3]:
served = from_pretrained("aimnet2", device=DEVICE)
bare = from_pretrained("aimnet2", wrap=False, dispersion=False, device=DEVICE)
core = served.model.model
print(type(served.model).__name__, "around", type(core).__name__,
      "| neighbor-list cutoff", served.cutoff, "A | local cutoff", core.aev_cutoff, "A")
print("species:", core.species)
print(f"features {core.n_features} x shells {core.n_rbf}, passes {len(core.passes)}, "
      f"AIM {core.aim_size}, SR Coulomb cutoff {core.coulomb_sr_cutoff:.3f} A ({core.coulomb} sum)")
print("parameters:", sum(p.numel() for p in core.parameters()))
print("shifts dtype:", core.atom_ref.weight.dtype, "| network dtype:", core.embedding.weight.dtype)
from xnn.common.models import load_pretrained
cfg = load_pretrained("aimnet2", local_files_only=True).config
print("recorded dispersion:", {k: v for k, v in cfg.subtracted_dispersion.items() if k != "note"})


INFO:xnn.common.models.hub.checkpoint:adding back the dispersion recorded as subtracted from the training labels: {'name': 'd3', 's6': 1.0, 's8': 0.3908, 'a1': 0.566, 'a2': 3.128, 'damping': 'bj', 's9': 0.0, 'cutoff_pair': 15.0, 'switch_width_pair': 3.0, 'cutoff_cn': 15.0, 'cutoff_triple': 15.0, 'note': 'the published AIMNet2 model is served with this D3(BJ) term (its training labels carry no dispersion correction)'}


INFO:xnn.common.models.hub.checkpoint:adding back the dispersion recorded as subtracted from the training labels: {'name': 'd3', 's6': 1.0, 's8': 0.3908, 'a1': 0.566, 'a2': 3.128, 'damping': 'bj', 's9': 0.0, 'cutoff_pair': 15.0, 'switch_width_pair': 3.0, 'cutoff_cn': 15.0, 'cutoff_triple': 15.0, 'note': 'the published AIMNet2 model is served with this D3(BJ) term (its training labels carry no dispersion correction)'}


D3Dispersion around AIMNet2 | neighbor-list cutoff 15.0 A | local cutoff 5.0 A
species: [1, 5, 6, 7, 8, 9, 14, 15, 16, 17, 33, 34, 35, 53]
features 16 x shells 16, passes 3, AIM 256, SR Coulomb cutoff 4.600 A (simple sum)
parameters: 2192893
shifts dtype: torch.float64 | network dtype: torch.float32
recorded dispersion: {'name': 'd3', 's6': 1.0, 's8': 0.3908, 'a1': 0.566, 'a2': 3.128, 'damping': 'bj', 's9': 0.0, 'cutoff_pair': 15.0, 'switch_width_pair': 3.0, 'cutoff_cn': 15.0, 'cutoff_triple': 15.0}


The reference artifact itself: its architecture YAML and the mapping of its
tensors onto the xnn modules.


In [4]:
from xnn.common.models.hub.cache import resolve_cache_dir
from xnn.gnn.models import aimnet2_foundation as af

raw = resolve_cache_dir(None) / "aimnet2-wb97m-d3-0" / "raw"
pt = sorted(raw.glob("*.pt")) if raw.is_dir() else []
if not pt:
    # the raw file is removed after a single-member conversion; fetch it again for a look
    from xnn.common.data.hub._download import download_file
    card = model_card("aimnet2")
    pt = [download_file(card.url, raw / card.url.rsplit("/", 1)[-1], md5=card.md5, quiet=True)]
artifact = af.load_artifact(pt[0])
print(artifact["model_yaml"])
sd = artifact["state_dict"]
mapping = {"afv.weight": "embedding.weight (rows of the 14 species)", "aev.shifts_s": "rbf.centers",
           "aev.eta_s": "config gaussian_width", "aev.rc_s": "config cutoff",
           "conv_a.agh": "conv_a.weight", "conv_q.agh": "conv_q.weight",
           "mlps.<i>.<k>.*": "passes.<i>.<k>.*", "outputs.energy_mlp.mlp.<k>.*": "readout.<k>.*",
           "outputs.atomic_shift.shifts.weight": "atom_ref.weight (float64)",
           "outputs.srcoulomb.rc": "config coulomb_sr_cutoff"}
print(f"{'reference tensor':40s} {'shape':>16s}  xnn")
for k, v in sd.items():
    key = k
    if k.startswith("mlps."):
        key = "mlps.<i>.<k>.*"
    elif k.startswith("outputs.energy_mlp.mlp."):
        key = "outputs.energy_mlp.mlp.<k>.*"
    print(f"{k:40s} {str(tuple(v.shape)):>16s}  {mapping.get(key, '(unused duplicate basis)')}")


class: aimnet.models.aimnet2.AIMNet2
kwargs:
  nfeature: 16
  d2features: true
  ncomb_v: 12
  hidden:
  - - 512
    - 380
  - - 512
    - 380
  - - 512
    - 380
    - 380
  aim_size: 256
  aev:
    rc_s: 5.0
    nshifts_s: 16
  outputs:
    energy_mlp:
      class: aimnet.modules.Output
      kwargs:
        n_in: 256
        n_out: 1
        key_in: aim
        key_out: energy
        mlp:
          activation_fn: torch.nn.GELU
          last_linear: true
          hidden:
          - 128
          - 128
    atomic_shift:
      class: aimnet.modules.AtomicShift
      kwargs:
        key_in: energy
        key_out: energy
    atomic_sum:
      class: aimnet.modules.AtomicSum
      kwargs:
        key_in: energy
        key_out: energy
    srcoulomb:
      class: aimnet.modules.SRCoulomb
      kwargs:
        rc: 4.599999904632568
        key_in: charges
        key_out: energy
        envelope: exp

reference tensor                                    shape  xnn
aev.rc_s              

## 3. Test molecules

In [5]:
# the fidelity molecules, built in code: a neutral molecule, a cation, an anion,
# radicals (for the open-shell model), sulfur / chlorine chemistry, a palladium complex
def _perturbed(atoms, seed, scale=0.05):
    rng = np.random.default_rng(seed)
    return atoms.get_positions() + rng.normal(scale=scale, size=(len(atoms), 3))


def _case(name, atoms, seed, **info):
    return {"name": name, "numbers": atoms.get_atomic_numbers().tolist(),
            "pos": np.round(_perturbed(atoms, seed), 6).tolist(), **info}


def _hydronium():
    return Atoms("OHHH", positions=[[0.0, 0.0, 0.0], [0.98, 0.0, 0.1],
                                    [-0.49, 0.8487, 0.1], [-0.49, -0.8487, 0.1]])


def _acetate():
    return Atoms("CCOOHHH", positions=[
        [0.0, 0.0, 0.0], [1.52, 0.0, 0.0], [2.12, 1.1, 0.0], [2.12, -1.1, 0.0],
        [-0.37, 0.52, 0.88], [-0.37, 0.52, -0.88], [-0.37, -1.03, 0.0]])


def _pd_complex():
    pos = [[0.0, 0.0, 0.0], [2.3, 0.0, 0.0], [-2.3, 0.0, 0.0], [0.0, 2.05, 0.0], [0.0, -2.05, 0.0]]
    for sign in (1, -1):
        y = sign * 2.05
        pos += [[0.0, y + sign * 0.35, 0.95], [0.82, y + sign * 0.35, -0.47],
                [-0.82, y + sign * 0.35, -0.47]]
    return Atoms("PdClClNNHHHHHH", positions=pos)


CASES = [
    _case("water", molecule("H2O"), 1),
    _case("ethanol", molecule("CH3CH2OH"), 2),
    _case("hydronium", _hydronium(), 3, charge=1.0),
    _case("acetate", _acetate(), 4, charge=-1.0),
    _case("methyl-radical", molecule("CH3"), 5, mult=2.0),
    _case("hydroxyl-radical", molecule("OH"), 6, mult=2.0),
    _case("thiophene-like", molecule("C4H4S"), 7),
    _case("chloromethane", molecule("CH3Cl"), 8),
    _case("pd-complex", _pd_complex(), 9),
]
print(f"{len(CASES)} molecules:", ", ".join(c["name"] for c in CASES))


9 molecules: water, ethanol, hydronium, acetate, methyl-radical, hydroxyl-radical, thiophene-like, chloromethane, pd-complex


## 4. The parity sweep: bare networks

Member 0 of every family (and the four members of the general model), the
reference calculator with `needs_dispersion=False` against the xnn network
with `dispersion=False`, in float32 on the same device. Molecules outside a
family's elements, charged ones for the neutral-only `rxn` model and radicals
for the closed-shell models are skipped.


In [6]:
def reference_eval(calc, case, periodic_cell=None):
    data = {"coord": torch.tensor(case["pos"], dtype=torch.float32),
            "numbers": torch.tensor(case["numbers"], dtype=torch.long),
            "charge": torch.tensor(float(case.get("charge", 0.0))),
            "mult": torch.tensor(float(case.get("mult", 1.0)))}
    if periodic_cell is not None:
        data["cell"] = torch.tensor(periodic_cell, dtype=torch.float32)
    out = calc(data, forces=True)
    return (float(out["energy"]), out["forces"].detach().cpu().double().numpy(),
            out["charges"].detach().cpu().double().numpy())


def xnn_eval(model, case, cell=None):
    s = {"pos": np.array(case["pos"]), "atomic_numbers": case["numbers"],
         "total_charge": case.get("charge", 0.0), "spin_multiplicity": case.get("mult", 1.0)}
    if cell is not None:
        s["cell"], s["pbc"] = np.array(cell), [True, True, True]
    g = structure_to_graph(s, model.cutoff, device=DEVICE)
    g.compute_dtype = torch.float32
    out = model(g)
    return (float(out["energy"]), out["forces"].detach().cpu().double().numpy(),
            out["charges"].detach().cpu().double().numpy())


def applicable(case, name, species):
    if not set(case["numbers"]) <= set(species):
        return False
    if case.get("charge", 0.0) != 0.0 and "rxn" in name:
        return False
    return case.get("mult", 1.0) == 1.0 or "nse" in name


MEMBERS = ["aimnet2-wb97m-d3-0", "aimnet2-wb97m-d3-1", "aimnet2-wb97m-d3-2", "aimnet2-wb97m-d3-3",
           "aimnet2-b973c-d3-0", "aimnet2-b973c-2025-d3-0", "aimnet2-nse-0", "aimnet2-pd-0",
           "aimnet2-rxn-0"]
REF_NAMES = {n: n.rsplit("-", 1)[0] + "_" + n.rsplit("-", 1)[1] for n in MEMBERS}   # the package's keys

results = {}
for name in MEMBERS:
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        ref = AIMNet2Calculator(REF_NAMES[name], device=DEVICE, needs_dispersion=False)
        model = from_pretrained(name, dispersion=False, device=DEVICE)
    species = model.model.species
    dE, dF, dQ, n = 0.0, 0.0, 0.0, 0
    for case in CASES:
        if not applicable(case, name, species):
            continue
        e_r, f_r, q_r = reference_eval(ref, case)
        e_x, f_x, q_x = xnn_eval(model, case)
        dE = max(dE, abs(e_x - e_r) / len(case["numbers"]))
        dF = max(dF, np.abs(f_x - f_r).max())
        dQ = max(dQ, np.abs(q_x - q_r).max())
        n += 1
    results[name] = (n, dE, dF, dQ)
    del ref, model
    print(f"{name:26s} {n} molecules | max |dE|/atom {dE:.1e} eV | max |dF| {dF:.1e} eV/A | max |dq| {dQ:.1e} e")


aimnet2-wb97m-d3-0         6 molecules | max |dE|/atom 2.2e-07 eV | max |dF| 2.9e-06 eV/A | max |dq| 1.8e-07 e


aimnet2-wb97m-d3-1         6 molecules | max |dE|/atom 3.7e-07 eV | max |dF| 2.9e-06 eV/A | max |dq| 1.2e-07 e


aimnet2-wb97m-d3-2         6 molecules | max |dE|/atom 2.4e-07 eV | max |dF| 4.8e-06 eV/A | max |dq| 1.5e-07 e


aimnet2-wb97m-d3-3         6 molecules | max |dE|/atom 3.0e-07 eV | max |dF| 5.7e-06 eV/A | max |dq| 8.9e-08 e


aimnet2-b973c-d3-0         6 molecules | max |dE|/atom 4.2e-07 eV | max |dF| 8.6e-06 eV/A | max |dq| 8.9e-08 e


aimnet2-b973c-2025-d3-0    6 molecules | max |dE|/atom 5.7e-07 eV | max |dF| 3.8e-06 eV/A | max |dq| 8.9e-08 e


aimnet2-nse-0              8 molecules | max |dE|/atom 1.7e-07 eV | max |dF| 2.9e-06 eV/A | max |dq| 5.1e-07 e


aimnet2-pd-0               7 molecules | max |dE|/atom 6.7e-08 eV | max |dF| 4.3e-06 eV/A | max |dq| 1.1e-07 e


aimnet2-rxn-0              2 molecules | max |dE|/atom 3.1e-07 eV | max |dF| 5.7e-06 eV/A | max |dq| 1.2e-07 e


The differences are the float32 round-off of the two networks (about 1e-6 eV
per atom in the energies, 1e-6 eV/A in the forces, 1e-7 e in the charges):
the per-element energy shifts of thousands of eV stay float64 on both sides
and the pair sums are accumulated in float64. No family deviates
systematically, including the two-channel open-shell model on the radicals
and the palladium model without a Coulomb term.


## 5. The served models: adding the D3(BJ) term

The same comparison with dispersion on both sides: the reference's
`nvalchemiops` D3 kernel against xnn's `D3Dispersion` built from the
recorded parameters (15 A pair cutoff, 20% quintic switching window,
coordination numbers over the same list). xnn's D3 is an independent
implementation verified against `simple-dftd3`; its reference data agree
with the D3 tables for the 14 elements here, so the remaining difference is
the kernels' float32 arithmetic.


In [7]:
for name in ("aimnet2-wb97m-d3-0", "aimnet2-b973c-2025-d3-0", "aimnet2-pd-0", "aimnet2-rxn-0"):
    with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
        ref = AIMNet2Calculator(REF_NAMES[name], device=DEVICE)
        model = from_pretrained(name, device=DEVICE)
    species = model.model.model.species
    dE, dF, n = 0.0, 0.0, 0
    for case in CASES:
        if not applicable(case, name, species):
            continue
        e_r, f_r, _ = reference_eval(ref, case)
        e_x, f_x, _ = xnn_eval(model, case)
        dE = max(dE, abs(e_x - e_r) / len(case["numbers"]))
        dF = max(dF, np.abs(f_x - f_r).max())
        n += 1
    print(f"{name:26s} {n} molecules | with D3: max |dE|/atom {dE:.1e} eV | max |dF| {dF:.1e} eV/A")
    del ref, model


INFO:xnn.common.models.hub.checkpoint:adding back the dispersion recorded as subtracted from the training labels: {'name': 'd3', 's6': 1.0, 's8': 0.3908, 'a1': 0.566, 'a2': 3.128, 'damping': 'bj', 's9': 0.0, 'cutoff_pair': 15.0, 'switch_width_pair': 3.0, 'cutoff_cn': 15.0, 'cutoff_triple': 15.0, 'note': 'the published AIMNet2 model is served with this D3(BJ) term (its training labels carry no dispersion correction)'}


aimnet2-wb97m-d3-0         6 molecules | with D3: max |dE|/atom 2.3e-07 eV | max |dF| 3.8e-06 eV/A


INFO:xnn.common.models.hub.checkpoint:adding back the dispersion recorded as subtracted from the training labels: {'name': 'd3', 's6': 1.0, 's8': 1.5, 'a1': 0.37, 'a2': 4.1, 'damping': 'bj', 's9': 0.0, 'cutoff_pair': 15.0, 'switch_width_pair': 3.0, 'cutoff_cn': 15.0, 'cutoff_triple': 15.0, 'note': 'the published AIMNet2 model is served with this D3(BJ) term (its training labels carry no dispersion correction)'}


aimnet2-b973c-2025-d3-0    6 molecules | with D3: max |dE|/atom 5.6e-07 eV | max |dF| 5.7e-06 eV/A


INFO:xnn.common.models.hub.checkpoint:adding back the dispersion recorded as subtracted from the training labels: {'name': 'd3', 's6': 1.0, 's8': 1.5, 'a1': 0.37, 'a2': 4.1, 'damping': 'bj', 's9': 0.0, 'cutoff_pair': 15.0, 'switch_width_pair': 3.0, 'cutoff_cn': 15.0, 'cutoff_triple': 15.0, 'note': 'the published AIMNet2 model is served with this D3(BJ) term (its training labels carry no dispersion correction)'}


aimnet2-pd-0               7 molecules | with D3: max |dE|/atom 8.5e-08 eV | max |dF| 4.3e-06 eV/A


INFO:xnn.common.models.hub.checkpoint:adding back the dispersion recorded as subtracted from the training labels: {'name': 'd3', 's6': 1.0, 's8': 0.3908, 'a1': 0.566, 'a2': 3.128, 'damping': 'bj', 's9': 0.0, 'cutoff_pair': 15.0, 'switch_width_pair': 3.0, 'cutoff_cn': 15.0, 'cutoff_triple': 15.0, 'note': 'the published AIMNet2 model is served with this D3(BJ) term (its training labels carry no dispersion correction)'}


aimnet2-rxn-0              2 molecules | with D3: max |dE|/atom 3.1e-07 eV | max |dF| 4.3e-06 eV/A


## 6. A periodic CO2 box with the damped shifted-force Coulomb sum

For periodic structures the all-pairs Coulomb sum does not apply; the paper's
condensed-phase CO2 simulation used the damped shifted-force (DSF) truncation
of Fennell and Gezelter at 15 A. The reference calculator switches to it with
`set_lrcoulomb_method("dsf", ...)`; in xnn the same model is served with
`model_options={"coulomb": "dsf", "lr_cutoff": ...}`, and the D3 cutoff is
set through the `dispersion` overrides. A 64-molecule box of 20 A at two
cutoffs.


In [8]:
from scipy.spatial.transform import Rotation

def co2_box(n_side=4, spacing=5.0, seed=11):
    rng = np.random.default_rng(seed)
    co2 = molecule("CO2")
    box = Atoms(cell=np.eye(3) * n_side * spacing, pbc=True)
    for i in range(n_side):
        for j in range(n_side):
            for k in range(n_side):
                m = co2.copy()
                m.set_positions(Rotation.random(random_state=rng.integers(1 << 31)).apply(m.get_positions()))
                m.translate(np.array([i, j, k]) * spacing + spacing / 2 + rng.normal(scale=0.3, size=3))
                box += m
    return box

box = co2_box()
case = {"name": "co2-box", "numbers": box.get_atomic_numbers().tolist(), "pos": box.get_positions().tolist()}
cell = np.asarray(box.get_cell())
print(len(box), "atoms in a", cell[0, 0], "A box")
for cutoff in (15.0, 9.0):
    for with_d3 in (False, True):
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            ref = AIMNet2Calculator("aimnet2-wb97m-d3_0", device=DEVICE, needs_dispersion=with_d3)
            ref.set_lrcoulomb_method("dsf", cutoff=cutoff, dsf_alpha=0.2)
            if with_d3:
                ref.set_dftd3_cutoff(cutoff, 0.2)
            disp = ({"cutoff_pair": cutoff, "switch_width_pair": 0.2 * cutoff, "cutoff_cn": cutoff,
                     "cutoff_triple": cutoff}
                    if with_d3 else False)
            model = from_pretrained("aimnet2", device=DEVICE, dispersion=disp,
                                    model_options={"coulomb": "dsf", "lr_cutoff": cutoff})
        try:
            e_r, f_r, q_r = reference_eval(ref, case, periodic_cell=cell)
        except Exception as exc:
            print(f"DSF {cutoff:4.1f} A, D3 {with_d3!s:5s}: reference failed ({type(exc).__name__}: {exc})")
            continue
        e_x, f_x, q_x = xnn_eval(model, case, cell=cell)
        print(f"DSF {cutoff:4.1f} A, D3 {with_d3!s:5s}: E_ref {e_r:.4f} eV, E_xnn {e_x:.4f} eV | "
              f"|dE|/atom {abs(e_x - e_r) / len(box):.1e} eV | max |dF| {np.abs(f_x - f_r).max():.1e} eV/A | "
              f"max |dq| {np.abs(q_x - q_r).max():.1e} e")
        del ref, model


192 atoms in a 20.0 A box


DSF 15.0 A, D3 False: E_ref -328648.9383 eV, E_xnn -328648.9382 eV | |dE|/atom 2.7e-07 eV | max |dF| 1.2e-04 eV/A | max |dq| 6.6e-07 e


INFO:xnn.common.models.hub.checkpoint:adding back the dispersion recorded as subtracted from the training labels, with overrides {'cutoff_pair': 15.0, 'switch_width_pair': 3.0, 'cutoff_cn': 15.0, 'cutoff_triple': 15.0}: {'name': 'd3', 's6': 1.0, 's8': 0.3908, 'a1': 0.566, 'a2': 3.128, 'damping': 'bj', 's9': 0.0, 'cutoff_pair': 15.0, 'switch_width_pair': 3.0, 'cutoff_cn': 15.0, 'cutoff_triple': 15.0, 'note': 'the published AIMNet2 model is served with this D3(BJ) term (its training labels carry no dispersion correction)'}


DSF 15.0 A, D3 True : E_ref -328653.8094 eV, E_xnn -328653.8093 eV | |dE|/atom 2.7e-07 eV | max |dF| 1.2e-04 eV/A | max |dq| 6.3e-07 e


DSF  9.0 A, D3 False: E_ref -328647.0303 eV, E_xnn -328647.0303 eV | |dE|/atom 2.8e-07 eV | max |dF| 1.2e-04 eV/A | max |dq| 6.6e-07 e


INFO:xnn.common.models.hub.checkpoint:adding back the dispersion recorded as subtracted from the training labels, with overrides {'cutoff_pair': 9.0, 'switch_width_pair': 1.8, 'cutoff_cn': 9.0, 'cutoff_triple': 9.0}: {'name': 'd3', 's6': 1.0, 's8': 0.3908, 'a1': 0.566, 'a2': 3.128, 'damping': 'bj', 's9': 0.0, 'cutoff_pair': 9.0, 'switch_width_pair': 1.8, 'cutoff_cn': 9.0, 'cutoff_triple': 9.0, 'note': 'the published AIMNet2 model is served with this D3(BJ) term (its training labels carry no dispersion correction)'}


DSF  9.0 A, D3 True : E_ref -328651.7617 eV, E_xnn -328651.7616 eV | |dE|/atom 2.6e-07 eV | max |dF| 1.2e-04 eV/A | max |dq| 6.9e-07 e


## 7. Ewald and particle-mesh Ewald

The reference calculator also offers the Ewald and particle-mesh Ewald (PME)
sums through `set_lrcoulomb_method("ewald" | "pme", ewald_accuracy=...)`,
backed by `nvalchemiops`; xnn serves the same model with
`model_options={"coulomb": "ewald" | "pme"}` from its own kernels
(`xnn.common.models.electrostatics`, `xnn.common.models.pme`). Both pick the
splitting parameter and the reciprocal cutoff by the Kolafa-Perram balance for
a target accuracy; the one difference is the real-space cutoff, which the
reference chooses per structure from the accuracy while xnn keeps it at
`lr_cutoff`, the radius of the graph the data pipeline builds.

Two layers of comparison. First the kernels alone, on fixed charges, with
identical parameters: xnn's splitting parameter, its reciprocal sphere handed
to the reference kernel as explicit lattice vectors, the same neighbor list,
and the same mesh and spline order for PME. In float32 and float64.


In [9]:
from ase.neighborlist import neighbor_list as ase_neighbor_list
from nvalchemiops.torch.interactions.electrostatics import ewald_summation, particle_mesh_ewald, ewald_real_space
from xnn.common.models.electrostatics import (COULOMB_CONSTANT, coulomb_ewald, coulomb_pme, ewald_mesh_sizes,
                                              ewald_parameters, ewald_real, reciprocal_lattice)
from xnn.common.data.neighborlist import build_neighbor_list

KE_REF = 27.211386245988 * 0.5291772105638411      # the reference's ASE constants (1e-8 relative from xnn's CODATA)

def rock_salt(a=5.64):
    base = np.array([[0, 0, 0], [0.5, 0.5, 0], [0.5, 0, 0.5], [0, 0.5, 0.5]], dtype=float)
    return np.concatenate([base, base + [0.5, 0, 0]]) * a, np.array([1.0] * 4 + [-1.0] * 4), np.eye(3) * a

def random_cell(seed=3, charged=False):
    rng = np.random.default_rng(seed)
    pos, q = rng.uniform(0.0, 9.0, (20, 3)), rng.normal(size=20)
    q = q + (1.0 if charged else 0.0) * (np.arange(20) == 0) - (0.0 if charged else q.mean())
    return pos, q, np.diag([9.0, 10.0, 11.0]) + rng.normal(scale=0.4, size=(3, 3))

box = co2_box()
KERNEL_CASES = {"rock salt (8)": rock_salt(), "random neutral (20)": random_cell(),
                "random charged (20)": random_cell(charged=True),
                "CO2 box, fixed charges (192)": (box.get_positions(), np.where(box.get_atomic_numbers() == 6, 0.7, -0.35),
                                                 np.asarray(box.get_cell()))}

def neighbor_matrix(pos, cell, cutoff, flip):
    i, j, s = ase_neighbor_list("ijS", Atoms(positions=pos, cell=cell, pbc=True), cutoff)
    n, counts = len(pos), np.bincount(i, minlength=len(pos))
    mat, shifts = np.full((n, counts.max()), n, dtype=np.int32), np.zeros((n, counts.max(), 3), dtype=np.int32)
    order = np.argsort(i, kind="stable"); i, j, s = i[order], j[order], s[order]
    starts = np.concatenate([[0], np.cumsum(counts)[:-1]])
    for a in range(n):
        mat[a, :counts[a]] = j[starts[a]:starts[a] + counts[a]]
        shifts[a, :counts[a]] = (-s if flip else s)[starts[a]:starts[a] + counts[a]]
    return torch.tensor(mat, device=DEVICE), torch.tensor(shifts, device=DEVICE)

def half_space(kvec, grid):
    h, k, m = grid.T
    return kvec[(h > 0) | ((h == 0) & (k > 0)) | ((h == 0) & (k == 0) & (m > 0))]

def xnn_pairs(pos, cell, cutoff):
    ei, sh = build_neighbor_list(pos, cutoff, cell, torch.ones(3, dtype=torch.bool, device=DEVICE))
    return ei, torch.linalg.norm(pos[ei[1]] - pos[ei[0]] + sh.to(pos.dtype) @ cell, dim=-1)

print(f"{'case':30s} {'dtype':7s} {'rc':>4s} {'eps':>5s} | {'E_xnn ewald (eV)':>17s} {'dE ewald':>9s} {'dF ewald':>9s} | {'dE pme':>9s} {'dF pme':>9s} | {'ewald-pme':>9s}")
for name, (pos_np, q_np, cell_np) in KERNEL_CASES.items():
    for dtype in (torch.float32, torch.float64):
        for cutoff, eps in ((6.0, 1e-6), (9.0, 1e-8)) if len(pos_np) < 100 else ((15.0, 1e-6),):
            alpha, k_cutoff = ewald_parameters(cutoff, eps)
            pos = torch.tensor(pos_np, dtype=dtype, device=DEVICE, requires_grad=True)
            q = torch.tensor(q_np, dtype=dtype, device=DEVICE)
            cell = torch.tensor(cell_np, dtype=dtype, device=DEVICE)
            ei, r = xnn_pairs(pos, cell, cutoff)
            e_x = coulomb_ewald(q, pos, cell, ei, r, alpha, cutoff, k_cutoff).sum()
            f_x = -torch.autograd.grad(e_x, pos, retain_graph=True)[0]
            mesh = ewald_mesh_sizes(cell, alpha, eps)
            e_p = coulomb_pme(q, pos, cell, ei, r, alpha, cutoff, eps, 4).sum()
            f_p = -torch.autograd.grad(e_p, pos)[0]
            # the shift convention of the reference's neighbor matrix: whichever reproduces the real-space sum
            e_real = float(ewald_real(q, ei, r, alpha, cutoff).sum())
            for flip in (False, True):
                mat, sh = neighbor_matrix(pos_np, cell_np, cutoff, flip)
                e_ref_real = ewald_real_space(positions=pos.detach(), charges=q, cell=cell, alpha=torch.tensor(alpha, dtype=dtype, device=DEVICE),
                                              neighbor_matrix=mat, neighbor_matrix_shifts=sh, mask_value=len(pos_np))
                if abs(float(e_ref_real.sum()) - e_real) < 1e-4 * max(1.0, abs(e_real)):
                    break
            grid, kvec, _ = reciprocal_lattice(cell.double(), k_cutoff)
            kv = half_space(kvec, grid).to(dtype)
            common = dict(neighbor_matrix=mat, neighbor_matrix_shifts=sh, mask_value=len(pos_np))
            p_ref = pos.detach().clone().requires_grad_(True)
            e_r = ewald_summation(positions=p_ref, charges=q, cell=cell, alpha=alpha, k_vectors=kv, **common).sum() * KE_REF
            f_r = -torch.autograd.grad(e_r, p_ref)[0]
            p_ref = pos.detach().clone().requires_grad_(True)
            e_rp = particle_mesh_ewald(positions=p_ref, charges=q, cell=cell, alpha=alpha, mesh_dimensions=tuple(mesh),
                                       spline_order=4, **common).sum() * KE_REF
            f_rp = -torch.autograd.grad(e_rp, p_ref)[0]
            print(f"{name:30s} {str(dtype)[6:]:7s} {cutoff:4.0f} {eps:5.0e} | {float(e_x):17.8f} {float(e_x - e_r):9.1e} "
                  f"{float((f_x - f_r).abs().max()):9.1e} | {float(e_p - e_rp):9.1e} {float((f_p - f_rp).abs().max()):9.1e} | "
                  f"{float(e_x - e_p):9.1e}")
            if name.startswith("rock salt") and dtype == torch.float64 and eps == 1e-8:
                exact = -4.0 * 1.747564594633182 * COULOMB_CONSTANT / 2.82
                print(f"{'':30s} rock salt, exact Madelung energy {exact:.8f} eV: xnn {float(e_x) - exact:+.1e}, "
                      f"reference {float(e_r) - exact:+.1e} (box lattice {float(ewald_summation(positions=pos.detach(), charges=q, cell=cell, alpha=alpha, k_cutoff=k_cutoff, **common).sum()) * KE_REF - exact:+.1e})")
                from xnn.common.models.electrostatics import ewald_corrections, ewald_reciprocal
                from xnn.common.models.ops import cell_volume
                from nvalchemiops.torch.interactions.electrostatics import ewald_reciprocal_space
                rec_x = float((ewald_reciprocal(q, pos.detach(), cell, alpha, k_cutoff) + ewald_corrections(q, alpha, cell_volume(cell))).sum()) * COULOMB_CONSTANT
                rec_r = ewald_reciprocal_space(positions=pos.detach(), charges=q, cell=cell, alpha=torch.tensor(alpha, dtype=dtype, device=DEVICE), k_vectors=kv)
                rec_r = float((rec_r[0] if isinstance(rec_r, tuple) else rec_r).sum()) * KE_REF
                print(f"{'':30s} components: real-space xnn - ref {e_real * COULOMB_CONSTANT - float(e_ref_real.sum()) * KE_REF:+.1e} eV, "
                      f"reciprocal + corrections xnn - ref {rec_x - rec_r:+.1e} eV")


case                           dtype     rc   eps |  E_xnn ewald (eV)  dE ewald  dF ewald |    dE pme    dF pme | ewald-pme


rock salt (8)                  float32    6 1e-06 |      -35.69405781  -1.0e-05   3.7e-07 |  -2.0e-05   6.4e-07 |  -7.0e-06


rock salt (8)                  float32    9 1e-08 |      -35.69405695  -2.0e-05   3.1e-07 |  -2.3e-05   5.9e-07 |  -1.8e-10


rock salt (8)                  float64    6 1e-06 |      -35.69405766  -1.2e-05   2.1e-16 |  -1.7e-05   1.2e-15 |  -7.0e-06
rock salt (8)                  float64    9 1e-08 |      -35.69405757  -2.2e-05   3.0e-16 |  -2.2e-05   3.9e-16 |  -1.8e-10
                               rock salt, exact Madelung energy -35.69405761 eV: xnn +4.0e-08, reference +2.2e-05 (box lattice +2.2e-05)
                               components: real-space xnn - ref -2.2e-05 eV, reciprocal + corrections xnn - ref -1.9e-08 eV


random neutral (20)            float32    6 1e-06 |      -67.99475478   4.8e-06   3.8e-06 |   1.4e-04   1.2e-04 |   1.7e-05


random neutral (20)            float32    9 1e-08 |      -67.99473612   8.0e-06   7.6e-06 |   8.5e-06   1.4e-04 |   5.0e-07
random neutral (20)            float64    6 1e-06 |      -67.99476000  -1.4e-06   4.3e-06 |  -3.6e-05   2.7e-06 |   1.7e-05


random neutral (20)            float64    9 1e-08 |      -67.99473915   4.2e-06   2.8e-06 |   3.9e-06   2.8e-06 |   5.0e-07
random charged (20)            float32    6 1e-06 |      -70.89266802   5.6e-06   2.9e-06 |   1.1e-04   1.2e-04 |   1.8e-05
random charged (20)            float32    9 1e-08 |      -70.89264557   4.1e-06   5.7e-06 |   8.2e-06   1.9e-04 |   6.9e-07
random charged (20)            float64    6 1e-06 |      -70.89266663   1.6e-06   3.6e-06 |  -3.8e-05   5.0e-06 |   1.8e-05


random charged (20)            float64    9 1e-08 |      -70.89264284   1.7e-06   3.5e-06 |   1.2e-06   3.6e-06 |   6.6e-07


CO2 box, fixed charges (192)   float32   15 1e-06 |     -335.50600783   3.7e-05   3.3e-06 |   3.4e-05   3.8e-06 |  -3.9e-07


CO2 box, fixed charges (192)   float64   15 1e-06 |     -335.50602134   4.0e-05   2.6e-07 |   3.9e-05   3.3e-07 |  -3.9e-07


With the same lattice vectors the two Ewald implementations agree to about
1e-6 relative in both precisions, and the residual is on the reference side:
xnn reproduces the exact Madelung energy of rock salt to 4e-8 eV while the
reference kernels land 2e-5 eV away, with either lattice set and for PME too,
and the component split places that difference in the reference's
real-space term (its reciprocal sum and corrections match xnn's to 2e-8 eV;
the reference script checked the same real-space kernel against a direct
sum of the pairs with the same result). The forces agree to float64
round-off in the symmetric crystal and to a few 1e-6 eV/A in the random
cells. The PME energies differ at the level of the mesh interpolation error:
xnn uses the exact Euler-spline coefficients of Essmann *et al.*, the
reference the continuous `sinc^p` deconvolution, two approximations of the
same reciprocal sum that agree to the target accuracy, as both do with the
Ewald sum (last column).

Second, the served model end to end: the reference calculator with
`set_lrcoulomb_method` against xnn's `model_options`, on the CO2 box, with
the reference's own automatic parameters. Float32 first, then float64
(the reference has no precision switch; casting its loaded model and
external modules to double is attempted here, xnn takes
`dtype=torch.float64`).


In [10]:
def reference_periodic(name, method, dtype, with_d3=False, cutoff=15.0):
    calc = AIMNet2Calculator(name, device=DEVICE, needs_dispersion=with_d3)
    if dtype == torch.float64:
        calc.model = calc.model.double()
        for mod in (calc.external_coulomb, calc.external_dftd3):
            if mod is not None:
                mod.double()
        calc.keys_in = {k: (torch.float64 if v == torch.float32 else v) for k, v in calc.keys_in.items()}
        calc.keys_in_optional = {k: (torch.float64 if v == torch.float32 else v) for k, v in calc.keys_in_optional.items()}
    if method == "dsf":
        calc.set_lrcoulomb_method("dsf", cutoff=cutoff, dsf_alpha=0.2)
    else:
        calc.set_lrcoulomb_method(method, ewald_accuracy=1e-6)
    return calc

def eval_periodic(calc, atoms, dtype):
    data = {"coord": torch.tensor(atoms.get_positions(), dtype=dtype), "numbers": torch.tensor(atoms.get_atomic_numbers()),
            "charge": torch.tensor(0.0, dtype=dtype), "mult": torch.tensor(1.0, dtype=dtype),
            "cell": torch.tensor(np.asarray(atoms.get_cell()), dtype=dtype)}
    t0 = time.time(); out = calc(data, forces=True)
    return float(out["energy"]), out["forces"].detach().cpu().double().numpy(), out["charges"].detach().cpu().double().numpy(), time.time() - t0

def xnn_periodic(model, atoms, dtype):
    s = {"pos": atoms.get_positions(), "atomic_numbers": atoms.get_atomic_numbers(), "cell": np.asarray(atoms.get_cell()), "pbc": [True] * 3}
    g = structure_to_graph(s, model.cutoff, device=DEVICE); g.compute_dtype = dtype
    t0 = time.time(); out = model(g)
    return float(out["energy"]), out["forces"].detach().cpu().double().numpy(), out["charges"].detach().cpu().double().numpy(), time.time() - t0

box = co2_box()
end_to_end, reference32 = {}, {}
for dtype in (torch.float32, torch.float64):
    for method in ("dsf", "ewald", "pme"):
        with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
            model = from_pretrained("aimnet2", device=DEVICE, dispersion=False, dtype=dtype,
                                    model_options={"coulomb": method, "lr_cutoff": 15.0})
        e_x, f_x, q_x, t_x = xnn_periodic(model, box, dtype)
        end_to_end[(str(dtype)[6:], method)] = (e_x, f_x, q_x)
        try:
            with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
                ref = reference_periodic("aimnet2-wb97m-d3_0", method, dtype)
            e_r, f_r, q_r, t_r = eval_periodic(ref, box, dtype)
            reference32[method] = (e_r, f_r, q_r)
            del ref
        except TypeError as exc:
            # the reference network is float32 only: compare against its float32 result instead
            e_r, f_r, q_r = reference32[method]
            t_r = float("nan")
            note = f"  (reference: {exc}; float32 reference used)"
        else:
            note = ""
        print(f"{str(dtype)[6:]} {method:5s}: E_ref {e_r:.5f} eV, E_xnn {e_x:.5f} eV | |dE|/atom {abs(e_x - e_r) / len(box):.1e} eV | "
              f"max |dF| {np.abs(f_x - f_r).max():.1e} eV/A | max |dq| {np.abs(q_x - q_r).max():.1e} e | "
              f"{t_r:.2f} s ref, {t_x:.2f} s xnn{note}")
        del model
print()
for p in ("float32", "float64"):
    print(f"{p}: xnn E(ewald) - E(pme) = {end_to_end[(p, 'ewald')][0] - end_to_end[(p, 'pme')][0]:+.2e} eV, "
          f"E(dsf 15 A) - E(ewald) = {end_to_end[(p, 'dsf')][0] - end_to_end[(p, 'ewald')][0]:+.2e} eV")
for method in ("dsf", "ewald", "pme"):
    e32, f32, q32 = end_to_end[("float32", method)]
    e64, f64, q64 = end_to_end[("float64", method)]
    print(f"xnn float64 vs float32, {method:5s}: |dE|/atom {abs(e64 - e32) / len(box):.1e} eV, max |dF| {np.abs(f64 - f32).max():.1e} eV/A, "
          f"max |dq| {np.abs(q64 - q32).max():.1e} e")


float32 dsf  : E_ref -328648.93828 eV, E_xnn -328648.93823 eV | |dE|/atom 2.4e-07 eV | max |dF| 1.2e-04 eV/A | max |dq| 6.9e-07 e | 0.02 s ref, 0.02 s xnn


float32 ewald: E_ref -328648.91366 eV, E_xnn -328648.91362 eV | |dE|/atom 2.5e-07 eV | max |dF| 1.2e-04 eV/A | max |dq| 6.3e-07 e | 0.11 s ref, 0.02 s xnn


float32 pme  : E_ref -328648.91366 eV, E_xnn -328648.91362 eV | |dE|/atom 2.1e-07 eV | max |dF| 1.2e-04 eV/A | max |dq| 6.6e-07 e | 0.18 s ref, 0.03 s xnn


float64 dsf  : E_ref -328648.93828 eV, E_xnn -328648.93822 eV | |dE|/atom 2.8e-07 eV | max |dF| 1.2e-04 eV/A | max |dq| 6.0e-07 e | nan s ref, 0.04 s xnn  (reference: conv_sv_2d_sp supports float32 tensors only; float32 reference used)


float64 ewald: E_ref -328648.91366 eV, E_xnn -328648.91361 eV | |dE|/atom 2.6e-07 eV | max |dF| 1.2e-04 eV/A | max |dq| 6.3e-07 e | nan s ref, 0.04 s xnn  (reference: conv_sv_2d_sp supports float32 tensors only; float32 reference used)


float64 pme  : E_ref -328648.91366 eV, E_xnn -328648.91361 eV | |dE|/atom 2.2e-07 eV | max |dF| 1.2e-04 eV/A | max |dq| 6.3e-07 e | nan s ref, 0.05 s xnn  (reference: conv_sv_2d_sp supports float32 tensors only; float32 reference used)

float32: xnn E(ewald) - E(pme) = +3.72e-07 eV, E(dsf 15 A) - E(ewald) = -2.46e-02 eV
float64: xnn E(ewald) - E(pme) = -1.92e-07 eV, E(dsf 15 A) - E(ewald) = -2.46e-02 eV
xnn float64 vs float32, dsf  : |dE|/atom 4.0e-08 eV, max |dF| 1.0e-05 eV/A, max |dq| 4.3e-07 e
xnn float64 vs float32, ewald: |dE|/atom 8.5e-09 eV, max |dF| 9.6e-06 eV/A, max |dq| 4.0e-07 e
xnn float64 vs float32, pme  : |dE|/atom 1.1e-08 eV, max |dF| 1.3e-05 eV/A, max |dq| 4.5e-07 e


End to end the lattice sums agree to the requested accuracy times the
Coulomb energy of the box: the two codes split the sum at different
real-space cutoffs (the reference's Kolafa-Perram choice for this box is
about 17 A, xnn's `lr_cutoff` is 15 A) and the float32 network adds its
own round-off, as in the molecular comparisons above. The reference network
cannot run in float64 (its Warp convolution kernel accepts float32 only), so
the float64 rows compare xnn's float64 model with the float32 reference, and
the last lines xnn's two precisions with each other: the float32 network
differs from its float64 twin by about 1e-6 eV per atom, while the Coulomb
sums themselves are float64 in both. The DSF truncation at 15 A differs from
the converged lattice sums by much more than the accuracy, which is why the
exact sums are worth having.


## 8. The TorchScript artifact

`export_torchscript_potential` scripts the served model (the network, its
Ewald/PME or DSF sum and the D3 term) into a self-contained `.pt` that
`torch.jit.load` runs without xnn. The net charge and spin multiplicity are
fixed at export. The artifact against the eager model on a charged molecule
and on the periodic box.


In [11]:
import tempfile
from xnn.common.deploy import export_torchscript_potential

acetate = next(c for c in CASES if c["name"] == "acetate")
with contextlib.redirect_stdout(io.StringIO()), contextlib.redirect_stderr(io.StringIO()):
    molecular = from_pretrained("aimnet2", device=DEVICE, wrap=False)
    periodic = from_pretrained("aimnet2", device=DEVICE, wrap=False, model_options={"coulomb": "ewald", "lr_cutoff": 15.0})
with tempfile.TemporaryDirectory() as tmp:
    for label, model, charge, case, cell in (("acetate anion, all pairs", molecular, -1.0, acetate, None),
                                              ("CO2 box, Ewald + D3", periodic, 0.0,
                                               {"numbers": box.get_atomic_numbers().tolist(), "pos": box.get_positions().tolist()},
                                               np.asarray(box.get_cell()))):
        path = f"{tmp}/aimnet2.pt"
        export_torchscript_potential(model, model.cutoff, path, total_charge=charge, metadata={"model": "aimnet2"})
        extra = {"cutoff": "", "dispersion": "", "charges": "", "total_charge": ""}
        artifact = torch.jit.load(path, map_location=DEVICE, _extra_files=extra)
        pos = torch.tensor(case["pos"], dtype=torch.float64, device=DEVICE)
        z = torch.tensor(case["numbers"], device=DEVICE)
        cell_t = torch.tensor(cell, dtype=torch.float64, device=DEVICE) if cell is not None else None
        pbc_t = torch.tensor([cell is not None] * 3, device=DEVICE)
        t0 = time.time(); out = artifact(pos, z, cell_t, pbc_t); t_ts = time.time() - t0
        s = {"pos": np.array(case["pos"]), "atomic_numbers": case["numbers"], "total_charge": charge}
        if cell is not None:
            s["cell"], s["pbc"] = cell, [True] * 3
        eager = ForceStressOutput(model, compute_stress=cell is not None)(structure_to_graph(s, model.cutoff, device=DEVICE))
        print(f"{label}: metadata {dict((k, v.decode()) for k, v in extra.items())}")
        print(f"   E artifact {float(out['energy']):.6f} eV vs eager {float(eager['energy']):.6f} eV | "
              f"max |dF| {float((out['forces'] - eager['forces']).abs().max()):.1e} eV/A | "
              f"max |dq| {float((out['charges'] - eager['charges']).abs().max()):.1e} e | sum q {float(out['charges'].sum()):+.4f} | {t_ts:.2f} s")
        if cell is not None:
            print(f"   stress: max |d sigma| {float((out['stress'] - eager['stress'][0]).abs().max()):.1e} eV/A^3")


INFO:xnn.common.models.hub.checkpoint:adding back the dispersion recorded as subtracted from the training labels: {'name': 'd3', 's6': 1.0, 's8': 0.3908, 'a1': 0.566, 'a2': 3.128, 'damping': 'bj', 's9': 0.0, 'cutoff_pair': 15.0, 'switch_width_pair': 3.0, 'cutoff_cn': 15.0, 'cutoff_triple': 15.0, 'note': 'the published AIMNet2 model is served with this D3(BJ) term (its training labels carry no dispersion correction)'}


INFO:xnn.common.models.hub.checkpoint:adding back the dispersion recorded as subtracted from the training labels: {'name': 'd3', 's6': 1.0, 's8': 0.3908, 'a1': 0.566, 'a2': 3.128, 'damping': 'bj', 's9': 0.0, 'cutoff_pair': 15.0, 'switch_width_pair': 3.0, 'cutoff_cn': 15.0, 'cutoff_triple': 15.0, 'note': 'the published AIMNet2 model is served with this D3(BJ) term (its training labels carry no dispersion correction)'}


acetate anion, all pairs: metadata {'cutoff': '15.0', 'dispersion': 'True', 'charges': 'True', 'total_charge': '-1.0'}
   E artifact -6221.649886 eV vs eager -6221.649888 eV | max |dF| 1.1e-06 eV/A | max |dq| 1.5e-07 e | sum q -1.0000 | 0.23 s


CO2 box, Ewald + D3: metadata {'cutoff': '15.0', 'dispersion': 'True', 'charges': 'True', 'total_charge': '0.0'}
   E artifact -328653.784707 eV vs eager -328653.784706 eV | max |dF| 3.7e-06 eV/A | max |dq| 3.0e-07 e | sum q +0.0000 | 0.20 s
   stress: max |d sigma| 3.0e-08 eV/A^3


## Summary

| | |
|---|---|
| checkpoints converted | all 24 published members of the six AIMNet2 families |
| bare-network parity vs `aimnet` | float32 round-off in energies, forces and charges on every family |
| served models (with D3) | the same, plus the two D3 kernels' float32 arithmetic |
| periodic (DSF Coulomb) | the same box, the same truncations, the same numbers |
| Ewald kernels (same lattice vectors) | round-off of the precision used, float32 and float64 |
| PME kernels | the mesh interpolation error, within the target accuracy; Ewald and PME agree likewise |
| Ewald/PME end to end | the target accuracy times the Coulomb energy, in both precisions |
| TorchScript artifact | the eager energies, forces, stress and charges, without xnn at run time |

Downstream of this notebook the converted models behave like any other xnn
model: `ForceStressOutput` for forces and stress, `XNNCalculator` for ASE
(which passes the net charge and spin multiplicity from `atoms.info` and
returns the partial charges), and fine-tuning through the standard `Trainer`
with `foundation: aimnet2` in the config
(see `examples/gnn/aimnet2/aimnet2_foundation_molecules.ipynb`).
